# Llibreries

In [53]:
import pandas as pd
import numpy as np
import json
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [54]:
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dimensions.head()

,Codi_Dimensio,Desc_Dimensio,Codi_Valor,Desc_Valor_CA,Desc_Valor_ES,Desc_Valor_EN
0,1,SEXE,1,Dona,Mujer,Female
1,1,SEXE,2,Home,Hombre,Male
2,2,EDAT_1,0,0 anys,0 años,0 years
3,2,EDAT_1,1,1 anys,1 años,1 years
4,2,EDAT_1,2,2 anys,2 años,2 years


In [55]:
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")
dim_barris.head()

,codi_districte,nom_districte,codi_barri,nom_barri,geometria_etrs89,geometria_wgs84
0,1,Ciutat Vella,1,el Raval,"POLYGON ((430162.1875 4581936.9845, 430102.838...","POLYGON ((2.16471378585589 41.3859301967194, 2..."
1,1,Ciutat Vella,2,el Barri Gòtic,"POLYGON ((431189.9075 4581851.4475, 431025.789...","POLYGON ((2.1770141884741 41.385248355328, 2.1..."
2,1,Ciutat Vella,3,la Barceloneta,"POLYGON ((432798.7341255 4582081.2599495, 4327...","POLYGON ((2.19622882469513 41.387454220446, 2...."
3,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera","POLYGON ((431733.736 4582441.816, 431557.5115 ...","POLYGON ((2.18345134701381 41.3906119681235, 2..."
4,2,Eixample,5,el Fort Pienc,"POLYGON ((431741.8152 4582625.6491, 432012.183...","POLYGON ((2.18352725722411 41.3922683849226, 2..."


# Dades Demogràfiques

En aquest apartat enriquirem els datasets amb algunes variables porcentuals que ens ajudaran a modelar segons el nostre objectiu.
- Població Total
- % població estrangera
- % estragners_occidentals (Amèrica del Nord i Europa) - Article Lopez Gay o Cocola Gant?
- % pct_universitaris/estudis_superiors
- % pct_sense_estudis


Finalment, eliminarem les columnes innecessaries i exportarem fitxer per a cada any net.

In [56]:
dimensions[dimensions["Desc_Dimensio"] == "NACIONALITAT_REGIO"]["Desc_Valor_CA"].unique()

array(['Àfrica oriental', 'Àfrica central', 'Àfrica septentrional',
       'Àfrica meridional', 'Àfrica occidental', 'Carib',
       'Amèrica central', 'Amèrica del sud', 'Amèrica del nord',
       'Àsia central', 'Àsia oriental', 'Àsia meridional',
       'Àsia sud-oriental', 'Àsia occidental', 'Europa oriental',
       'Europa septentrional', 'Europa meridional', 'Europa occidental',
       'Austràlia i Nova Zelanda', 'Melanèsia', 'Micronèsia', 'Polinèsia',
       'No consta'], dtype=object)

En el càlcul de població estrangera occidental considerarem només les regions d' Amèrica del nord i Europa. Per tant, els valors per al càlcul són:
- Amèrica del nord ()
- Resta de la Unió Europea (Població per barri per nacionalitat)

Aquest indicador ens permetrà estudiar com atribut individual la immigració de països amb major poder adquisitiu.

In [57]:
poblacio_23_raw = pd.read_csv("../data/processed/df_demografia_23.csv")
poblacio_23_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 44 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   codi_barri                     73 non-null     int64  
 1   poblacio_total                 73 non-null     int64  
 2   espanya                        73 non-null     int64  
 3   no_consta_x                    73 non-null     int64  
 4   resta_de_la_uni_europea        73 non-null     int64  
 5   resta_del_mn                   73 non-null     int64  
 6   amrica_central                 73 non-null     int64  
 7   amrica_del_nord                73 non-null     int64  
 8   amrica_del_sud                 73 non-null     int64  
 9   austrlia_i_nova_zelanda        73 non-null     int64  
 10  carib                          73 non-null     int64  
 11  europa_meridional              73 non-null     int64  
 12  europa_occidental              73 non-null     int64

In [58]:
poblacio_15_raw = pd.read_csv("../data/processed/df_demografia_15.csv")
poblacio_15_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 42 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   codi_barri                     73 non-null     int64  
 1   poblacio_total                 73 non-null     int64  
 2   espanya                        73 non-null     int64  
 3   no_consta_x                    73 non-null     int64  
 4   resta_de_la_uni_europea        73 non-null     int64  
 5   resta_del_mn                   73 non-null     int64  
 6   amrica_central                 73 non-null     int64  
 7   amrica_del_nord                73 non-null     int64  
 8   amrica_del_sud                 73 non-null     int64  
 9   austrlia_i_nova_zelanda        73 non-null     int64  
 10  carib                          73 non-null     int64  
 11  europa_meridional              73 non-null     int64  
 12  europa_occidental              73 non-null     int64

In [59]:
# Apilem primer per aplicar transformacions
poblacio = pd.concat([poblacio_23_raw, poblacio_15_raw], keys=[2023, 2015]).reset_index(names=["periode", "index"]).drop(columns=["index"])
poblacio.head()

,periode,codi_barri,poblacio_total,espanya,no_consta_x,resta_de_la_uni_europea,resta_del_mn,amrica_central,amrica_del_nord,amrica_del_sud,...,joves_espanya,joves_no_consta,joves_resta_de_la_uni_europea,joves_resta_del_mn,joves_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris
0,2023,1,45671,22396,56,4563,18693,470,297,2888,...,3253,29,2362,11771,17415,0.509623,0.106413,0.490377,0.381314,0.241860
1,2023,2,24460,8385,32,3281,12784,453,306,2203,...,1435,16,1667,9196,12314,0.657195,0.146648,0.342805,0.503434,0.278250
2,2023,3,14274,8264,4,2619,3390,184,129,1248,...,1465,4,1369,2992,5830,0.421045,0.192518,0.578955,0.408435,0.305240
3,2023,4,22041,11764,16,4284,5989,313,390,1736,...,2016,16,2189,5049,9270,0.466267,0.212059,0.533733,0.420580,0.406288
4,2023,5,34403,23354,16,3445,7600,608,151,3088,...,3851,20,1270,6043,11184,0.321164,0.104526,0.678836,0.325088,0.377961


In [60]:
# Calcul de percentatges
poblacio["pct_pob_estrangera"] = 1 -(poblacio["espanya"] / poblacio["poblacio_total"])
poblacio["pct_pob_estrangera_occidental"] = (poblacio["resta_de_la_uni_europea"] + poblacio["amrica_del_nord"]) / poblacio["poblacio_total"]
poblacio["pct_joves"] = poblacio["joves_total"] / poblacio["poblacio_total"]
poblacio["pct_universitaris"] = (poblacio["estudis_superiors_espanya"] + poblacio["estudis_superiors_ue"] + poblacio["estudis_superiors_mon"]) / poblacio["poblacio_total"]

In [61]:
# Seleccionem conjunt d' atributs finals
poblacio = poblacio[["codi_barri", "periode", "poblacio_total", "pct_pob_estrangera", "pct_pob_estrangera_occidental", "pct_pob_espanyola", "pct_joves", "pct_universitaris"]]
poblacio.head()

,codi_barri,periode,poblacio_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris
0,1,2023,45671,0.509623,0.106413,0.490377,0.381314,0.241860
1,2,2023,24460,0.657195,0.146648,0.342805,0.503434,0.278250
2,3,2023,14274,0.421045,0.192518,0.578955,0.408435,0.305240
3,4,2023,22041,0.466267,0.212059,0.533733,0.420580,0.406288
4,5,2023,34403,0.321164,0.104526,0.678836,0.325088,0.377961


In [62]:
# Separem les dades per 2023 i 2015 per posteriorment crear un conjunt de modelling per als dos
poblacio_15 = poblacio[poblacio["periode"] == 2015].drop(columns= ["periode"])
poblacio_23 = poblacio[poblacio["periode"] == 2023].drop(columns= ["periode"])

In [63]:
poblacio_pivot = poblacio.pivot(index="codi_barri", columns="periode")
poblacio_pivot.head()

poblacio_total        pct_pob_estrangera            \
periode              2015   2023               2015      2023   
codi_barri                                                      
1                   47150  45671           0.480424  0.509623   
2                   15514  24460           0.419428  0.657195   
3                   15037  14274           0.312961  0.421045   
4                   22468  22041           0.392736  0.466267   
5                   31548  34403           0.194339  0.321164   

           pct_pob_estrangera_occidental           pct_pob_espanyola  \
periode                             2015      2023              2015   
codi_barri                                                             
1                               0.095949  0.106413          0.519576   
2                               0.209037  0.146648          0.580572   
3                               0.151160  0.192518          0.687039   
4                               0.194632  0.212059          0.607264   
5                               0.076455  0.104526          0.805661   

                     pct_joves           pct_universitaris            
periode         2023      2015      2023              2015      2023  
codi_barri                                                            
1           0.490377  0.394040  0.381314          0.184963  0.241860  
2           0.342805  0.428129  0.503434          0.333570  0.278250  
3           0.578955  0.373213  0.408435          0.214737  0.305240  
4           0.533733  0.403819  0.420580          0.323393  0.406288  
5           0.678836  0.296215  0.325088          0.313966  0.377961

In [64]:
poblacio_pivot.columns = [f"{col}_{year}" for col, year in poblacio_pivot.columns]
poblacio_pivot = poblacio_pivot.reset_index()
poblacio_pivot.head()

,codi_barri,poblacio_total_2015,poblacio_total_2023,pct_pob_estrangera_2015,pct_pob_estrangera_2023,pct_pob_estrangera_occidental_2015,pct_pob_estrangera_occidental_2023,pct_pob_espanyola_2015,pct_pob_espanyola_2023,pct_joves_2015,pct_joves_2023,pct_universitaris_2015,pct_universitaris_2023
0,1,47150,45671,0.480424,0.509623,0.095949,0.106413,0.519576,0.490377,0.394040,0.381314,0.184963,0.241860
1,2,15514,24460,0.419428,0.657195,0.209037,0.146648,0.580572,0.342805,0.428129,0.503434,0.333570,0.278250
2,3,15037,14274,0.312961,0.421045,0.151160,0.192518,0.687039,0.578955,0.373213,0.408435,0.214737,0.305240
3,4,22468,22041,0.392736,0.466267,0.194632,0.212059,0.607264,0.533733,0.403819,0.420580,0.323393,0.406288
4,5,31548,34403,0.194339,0.321164,0.076455,0.104526,0.805661,0.678836,0.296215,0.325088,0.313966,0.377961


In [65]:
# Deltas
vars_delta = [
    "pct_pob_estrangera",
    "pct_pob_estrangera_occidental",
    "pct_joves",
    "pct_universitaris"
]

for var in vars_delta:
    poblacio_pivot[f"delta_{var}"] = poblacio_pivot[f"{var}_2023"] - poblacio_pivot[f"{var}_2015"]

poblacio_pivot["delta_pob_pct"] = (poblacio_pivot["poblacio_total_2023"] - poblacio_pivot["poblacio_total_2015"]
) / poblacio_pivot["poblacio_total_2015"]

poblacio_pivot.head()

,codi_barri,poblacio_total_2015,poblacio_total_2023,pct_pob_estrangera_2015,pct_pob_estrangera_2023,pct_pob_estrangera_occidental_2015,pct_pob_estrangera_occidental_2023,pct_pob_espanyola_2015,pct_pob_espanyola_2023,pct_joves_2015,pct_joves_2023,pct_universitaris_2015,pct_universitaris_2023,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_pob_pct
0,1,47150,45671,0.480424,0.509623,0.095949,0.106413,0.519576,0.490377,0.394040,0.381314,0.184963,0.241860,0.029199,0.010464,-0.012726,0.056897,-0.031368
1,2,15514,24460,0.419428,0.657195,0.209037,0.146648,0.580572,0.342805,0.428129,0.503434,0.333570,0.278250,0.237768,-0.062389,0.075305,-0.055319,0.576640
2,3,15037,14274,0.312961,0.421045,0.151160,0.192518,0.687039,0.578955,0.373213,0.408435,0.214737,0.305240,0.108084,0.041357,0.035222,0.090503,-0.050742
3,4,22468,22041,0.392736,0.466267,0.194632,0.212059,0.607264,0.533733,0.403819,0.420580,0.323393,0.406288,0.073531,0.017427,0.016761,0.082895,-0.019005
4,5,31548,34403,0.194339,0.321164,0.076455,0.104526,0.805661,0.678836,0.296215,0.325088,0.313966,0.377961,0.126825,0.028071,0.028873,0.063995,0.090497


In [66]:
# Ens quedem amb els deltas per al dataset final de poblacio
poblacio_final = poblacio_pivot[[col for col in poblacio_pivot.columns if col.startswith("delta") or col == "codi_barri"]]
poblacio_final.head()

,codi_barri,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_pob_pct
0,1,0.029199,0.010464,-0.012726,0.056897,-0.031368
1,2,0.237768,-0.062389,0.075305,-0.055319,0.576640
2,3,0.108084,0.041357,0.035222,0.090503,-0.050742
3,4,0.073531,0.017427,0.016761,0.082895,-0.019005
4,5,0.126825,0.028071,0.028873,0.063995,0.090497


# Dades Econòmiques
Incorporem dades econòmiques:
- Renda neta mitja per persona.
- Índex de gini

In [67]:
econs_15 = pd.read_csv("../data/processed/df_economiques_15.csv")
econs_15.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   codi_barri    73 non-null     int64  
 1   import_euros  73 non-null     float64
 2   index_gini    73 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 1.8 KB


In [68]:
econs_23 = pd.read_csv("../data/processed/df_economiques_23.csv")
econs_23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   codi_barri    73 non-null     int64  
 1   import_euros  73 non-null     float64
 2   index_gini    73 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 1.8 KB


In [69]:
# Combinem
econs = dim_barris[["codi_barri"]].copy()
econs_merged = econs.merge(econs_23, on="codi_barri", how="left")\
    .merge(econs_15, on = "codi_barri", suffixes=["_2023", "_2015"])

econs_merged.head()

,codi_barri,import_euros_2023,index_gini_2023,import_euros_2015,index_gini_2015
0,1,11917.476190,33.795238,8507.809524,36.338095
1,2,16013.222222,39.266667,11447.444444,41.087500
2,3,15359.818182,32.809091,10818.818182,34.954545
3,4,16621.153846,36.769231,11689.000000,39.515385
4,5,20418.800000,31.890000,15441.500000,34.020000


In [70]:
# Calculem deltes
econs_final = econs_merged.copy()
econs_final["delta_renda_pct"] = (econs_final["import_euros_2023"] - econs_final["import_euros_2015"]) / econs_final["import_euros_2015"]
econs_final["delta_gini_pct"] = (econs_final["index_gini_2023"] - econs_final["index_gini_2015"]) / econs_final["index_gini_2015"]

econs_final.head()

,codi_barri,import_euros_2023,index_gini_2023,import_euros_2015,index_gini_2015,delta_renda_pct,delta_gini_pct
0,1,11917.476190,33.795238,8507.809524,36.338095,0.400769,-0.069978
1,2,16013.222222,39.266667,11447.444444,41.087500,0.398847,-0.044316
2,3,15359.818182,32.809091,10818.818182,34.954545,0.419732,-0.061378
3,4,16621.153846,36.769231,11689.000000,39.515385,0.421948,-0.069496
4,5,20418.800000,31.890000,15441.500000,34.020000,0.322333,-0.062610


In [71]:
# Ens  quedem amb els deltes només
econs_final = econs_final[[col for col in econs_final.columns if col.startswith("delta") or col == "codi_barri"]]
econs_final.head()

,codi_barri,delta_renda_pct,delta_gini_pct
0,1,0.400769,-0.069978
1,2,0.398847,-0.044316
2,3,0.419732,-0.061378
3,4,0.421948,-0.069496
4,5,0.322333,-0.062610


# Dades Urbanes
En aquest cas les dades són en valors absoluts. Per tal de neutralitzar l'efecte esbiaixador provocat per barris amb més població, normalitzarem les dades a un càlcul de cada 1000 habitants.

In [72]:
df_urbanes_23_raw = pd.read_csv("../data/processed/df_urbanes_23.csv")
df_urbanes_23_raw.head()

,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals
0,1,748,500,8,10
1,2,700,440,6,2
2,3,651,177,15,5
3,4,676,387,13,2
4,5,638,200,23,33


In [73]:
df_urbanes_15_raw = pd.read_csv("../data/processed/df_urbanes_15.csv")
df_urbanes_15_raw.head()

,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals
0,1,674,544,7,33
1,2,625,517,4,5
2,3,615,218,8,6
3,4,592,374,8,22
4,5,564,174,16,55


In [74]:
# Combinem els dos datasets
df_urbanes = pd.concat([df_urbanes_23_raw, df_urbanes_15_raw], keys= [2023, 2015]).reset_index(names=["periode", "remove"]).drop(columns=["remove"])
df_urbanes.head()

,periode,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals
0,2023,1,748,500,8,10
1,2023,2,700,440,6,2
2,2023,3,651,177,15,5
3,2023,4,676,387,13,2
4,2023,5,638,200,23,33


In [75]:
poblacio

,codi_barri,periode,poblacio_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris
0,1,2023,45671,0.509623,0.106413,0.490377,0.381314,0.241860
1,2,2023,24460,0.657195,0.146648,0.342805,0.503434,0.278250
2,3,2023,14274,0.421045,0.192518,0.578955,0.408435,0.305240
3,4,2023,22041,0.466267,0.212059,0.533733,0.420580,0.406288
4,5,2023,34403,0.321164,0.104526,0.678836,0.325088,0.377961
...,...,...,...,...,...,...,...,...
141,69,2015,13374,0.167713,0.084492,0.832287,0.269702,0.306640
142,70,2015,22844,0.241114,0.026440,0.758886,0.287428,0.083567
143,71,2015,20163,0.124436,0.038734,0.875564,0.280067,0.186728
144,72,2015,25971,0.104116,0.023372,0.895884,0.235801,0.150052


In [76]:
# Obtenim dades de la població en barris per periode
df_urbanes_join = df_urbanes.merge(poblacio[["codi_barri", "periode", "poblacio_total"]], on = ["codi_barri", "periode"], how= "left")
df_urbanes_join.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   periode                       146 non-null    int64
 1   codi_barri                    146 non-null    int64
 2   total_incidents               146 non-null    int64
 3   locals_restauracio            146 non-null    int64
 4   locals_sanitaris              146 non-null    int64
 5   locals_serveis_professionals  146 non-null    int64
 6   poblacio_total                146 non-null    int64
dtypes: int64(7)
memory usage: 8.1 KB


- Info de tots els barris, no hi ha nuls

In [77]:
df_urbanes.columns

Index(['periode', 'codi_barri', 'total_incidents', 'locals_restauracio',
       'locals_sanitaris', 'locals_serveis_professionals'],
      dtype='object')

In [78]:
# Normalitzem valors
df_urbanes_metriques = df_urbanes_join.copy()
metriques = ['total_incidents', 'locals_restauracio','locals_sanitaris', 'locals_serveis_professionals']

for metrica in metriques:
    df_urbanes_metriques[f"{metrica}_1000_hab"] = df_urbanes_metriques[metrica] / df_urbanes_metriques["poblacio_total"] * 1000

df_urbanes_metriques.head()

,periode,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals,poblacio_total,total_incidents_1000_hab,locals_restauracio_1000_hab,locals_sanitaris_1000_hab,locals_serveis_professionals_1000_hab
0,2023,1,748,500,8,10,45671,16.378008,10.947866,0.175166,0.218957
1,2023,2,700,440,6,2,24460,28.618152,17.988553,0.245298,0.081766
2,2023,3,651,177,15,5,14274,45.607398,12.400168,1.050862,0.350287
3,2023,4,676,387,13,2,22041,30.670115,17.558187,0.589810,0.090740
4,2023,5,638,200,23,33,34403,18.544894,5.813447,0.668546,0.959219


In [79]:
# Seleccionem les columnes que acabem de crear
df_urbanes_clean = df_urbanes_metriques[[col for col in df_urbanes_metriques.columns if col.endswith("_hab") or col in ["codi_barri", "periode"]]]
df_urbanes_clean.head()

,periode,codi_barri,total_incidents_1000_hab,locals_restauracio_1000_hab,locals_sanitaris_1000_hab,locals_serveis_professionals_1000_hab
0,2023,1,16.378008,10.947866,0.175166,0.218957
1,2023,2,28.618152,17.988553,0.245298,0.081766
2,2023,3,45.607398,12.400168,1.050862,0.350287
3,2023,4,30.670115,17.558187,0.589810,0.090740
4,2023,5,18.544894,5.813447,0.668546,0.959219


In [80]:
# Separem els dos conjunts de dades
df_urbanes_23 = df_urbanes_clean[df_urbanes_clean["periode"] == 2023].drop(columns=["periode"])
df_urbanes_15 = df_urbanes_clean[df_urbanes_clean["periode"] == 2015].drop(columns=["periode"])

In [81]:
df_urbanes_pivot = df_urbanes_clean.pivot(index = "codi_barri", columns="periode")
df_urbanes_pivot.head()

total_incidents_1000_hab            locals_restauracio_1000_hab  \
periode                        2015       2023                        2015   
codi_barri                                                                   
1                         14.294804  16.378008                   11.537646   
2                         40.286193  28.618152                   33.324739   
3                         40.899116  45.607398                   14.497573   
4                         26.348585  30.670115                   16.645896   
5                         17.877520  18.544894                    5.515405   

                      locals_sanitaris_1000_hab            \
periode          2023                      2015      2023   
codi_barri                                                  
1           10.947866                  0.148462  0.175166   
2           17.988553                  0.257832  0.245298   
3           12.400168                  0.532021  1.050862   
4           17.558187                  0.356062  0.589810   
5            5.813447                  0.507164  0.668546   

           locals_serveis_professionals_1000_hab            
periode                                     2015      2023  
codi_barri                                                  
1                                       0.699894  0.218957  
2                                       0.322290  0.081766  
3                                       0.399016  0.350287  
4                                       0.979170  0.090740  
5                                       1.743375  0.959219

In [82]:
# Modifiquem noms de columnes
df_urbanes_pivot.columns = [f"{col}_{year}" for col, year in df_urbanes_pivot.columns]
df_urbanes_pivot = df_urbanes_pivot.reset_index()
df_urbanes_pivot.head()

,codi_barri,total_incidents_1000_hab_2015,total_incidents_1000_hab_2023,locals_restauracio_1000_hab_2015,locals_restauracio_1000_hab_2023,locals_sanitaris_1000_hab_2015,locals_sanitaris_1000_hab_2023,locals_serveis_professionals_1000_hab_2015,locals_serveis_professionals_1000_hab_2023
0,1,14.294804,16.378008,11.537646,10.947866,0.148462,0.175166,0.699894,0.218957
1,2,40.286193,28.618152,33.324739,17.988553,0.257832,0.245298,0.322290,0.081766
2,3,40.899116,45.607398,14.497573,12.400168,0.532021,1.050862,0.399016,0.350287
3,4,26.348585,30.670115,16.645896,17.558187,0.356062,0.589810,0.979170,0.090740
4,5,17.877520,18.544894,5.515405,5.813447,0.507164,0.668546,1.743375,0.959219


In [83]:
# Calculem deltes
deltes = ['total_incidents_1000_hab','locals_restauracio_1000_hab', 'locals_sanitaris_1000_hab','locals_serveis_professionals_1000_hab']

for delta in deltes:
    df_urbanes_pivot[f"delta_pct_{delta}"] = (df_urbanes_pivot[f"{delta}_2023"] - df_urbanes_pivot[f"{delta}_2015"]) / df_urbanes_pivot[f"{delta}_2015"]

df_urbanes_pivot.head()

,codi_barri,total_incidents_1000_hab_2015,total_incidents_1000_hab_2023,locals_restauracio_1000_hab_2015,locals_restauracio_1000_hab_2023,locals_sanitaris_1000_hab_2015,locals_sanitaris_1000_hab_2023,locals_serveis_professionals_1000_hab_2015,locals_serveis_professionals_1000_hab_2023,delta_pct_total_incidents_1000_hab,delta_pct_locals_restauracio_1000_hab,delta_pct_locals_sanitaris_1000_hab,delta_pct_locals_serveis_professionals_1000_hab
0,1,14.294804,16.378008,11.537646,10.947866,0.148462,0.175166,0.699894,0.218957,0.145732,-0.051118,0.179867,-0.687156
1,2,40.286193,28.618152,33.324739,17.988553,0.257832,0.245298,0.322290,0.081766,-0.289629,-0.460204,-0.048610,-0.746296
2,3,40.899116,45.607398,14.497573,12.400168,0.532021,1.050862,0.399016,0.350287,0.115119,-0.144673,0.975226,-0.122122
3,4,26.348585,30.670115,16.645896,17.558187,0.356062,0.589810,0.979170,0.090740,0.164014,0.054806,0.656481,-0.907330
4,5,17.877520,18.544894,5.515405,5.813447,0.507164,0.668546,1.743375,0.959219,0.037330,0.054038,0.318206,-0.449792


In [84]:
# Ens quedem amb els deltas per al dataset
urbanes_final = df_urbanes_pivot[[col for col in df_urbanes_pivot.columns if col.startswith("delta") or col == "codi_barri"]]
urbanes_final.head()

,codi_barri,delta_pct_total_incidents_1000_hab,delta_pct_locals_restauracio_1000_hab,delta_pct_locals_sanitaris_1000_hab,delta_pct_locals_serveis_professionals_1000_hab
0,1,0.145732,-0.051118,0.179867,-0.687156
1,2,-0.289629,-0.460204,-0.048610,-0.746296
2,3,0.115119,-0.144673,0.975226,-0.122122
3,4,0.164014,0.054806,0.656481,-0.907330
4,5,0.037330,0.054038,0.318206,-0.449792


# Dades habitatge
Seguint el procediment utilitzat amb les dades urbanes, les dades d'habitatge amb valors absoluts es normalitzaran per evitar els efectes de variabilitat de la poblaicó entre barris

In [85]:
df_habitatges_23_raw = pd.read_csv("../data/processed/df_habitatge_23.csv")
df_habitatges_23_raw.head()

,codi_barri,preu_mitja_m2,pisos_turistics
0,1,15.9,212.0
1,2,16.4,201.0
2,3,19.0,64.0
3,4,17.4,171.0
4,5,16.1,333.0


In [86]:
df_habitatges_15_raw = pd.read_csv("../data/processed/df_habitatge_15.csv")
df_habitatges_15_raw.head()

,codi_barri,preu_mitja_m2,pisos_turistics
0,1,11.1,180.0
1,2,11.2,184.0
2,3,16.3,69.0
3,4,12.7,171.0
4,5,10.9,343.0


In [87]:
# Apilem dades
df_habitatges_concat = pd.concat([df_habitatges_23_raw, df_habitatges_15_raw], keys = [2023, 2015]).reset_index(names = ["periode", "remove"]).drop(columns=["remove"])
df_habitatges_concat.head()

,periode,codi_barri,preu_mitja_m2,pisos_turistics
0,2023,1,15.9,212.0
1,2023,2,16.4,201.0
2,2023,3,19.0,64.0
3,2023,4,17.4,171.0
4,2023,5,16.1,333.0


In [88]:
 # Portem informació de les poblacions en cada barri i periode
df_habitatges_join = df_habitatges_concat.merge(poblacio[["codi_barri", "periode", "poblacio_total"]], on = ["codi_barri", "periode"], how= "left")
df_habitatges_join.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   periode          146 non-null    int64  
 1   codi_barri       146 non-null    int64  
 2   preu_mitja_m2    146 non-null    float64
 3   pisos_turistics  128 non-null    float64
 4   poblacio_total   146 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 5.8 KB


- Existeixen alguns nuls en barris on no hi ha pisos turístics

In [89]:
df_habitatges_join[df_habitatges_join["pisos_turistics"].isna()]

,periode,codi_barri,preu_mitja_m2,pisos_turistics,poblacio_total
39,2023,40,13.8,NaN,5174
41,2023,42,15.5,NaN,699
46,2023,47,12.5,NaN,2170
52,2023,53,13.5,NaN,7624
53,2023,54,7.7,NaN,2939
54,2023,55,10.4,NaN,10857
55,2023,56,8.0,NaN,1421
57,2023,58,9.2,NaN,2582
112,2015,40,10.1,NaN,5041
113,2015,41,10.1,NaN,5467


- Donat que es tracta de barris en els que no hi ha presència de pisos turístics, és correcte imputar-los amb un 0 i no amb mesures de tendència com la mitjana, mediana, etc.

In [90]:
df_habitatges_join["pisos_turistics"] = df_habitatges_join["pisos_turistics"].fillna(0)
df_habitatges_join.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   periode          146 non-null    int64  
 1   codi_barri       146 non-null    int64  
 2   preu_mitja_m2    146 non-null    float64
 3   pisos_turistics  146 non-null    float64
 4   poblacio_total   146 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 5.8 KB


In [91]:
# Creem metrica i comprovem
df_habitatges_metriques = df_habitatges_join.copy()
df_habitatges_metriques["pisos_turistics_1000_hab"] = df_habitatges_metriques["pisos_turistics"] / df_habitatges_metriques["poblacio_total"] * 1000
df_habitatges_metriques.head()

,periode,codi_barri,preu_mitja_m2,pisos_turistics,poblacio_total,pisos_turistics_1000_hab
0,2023,1,15.9,212.0,45671,4.641895
1,2023,2,16.4,201.0,24460,8.217498
2,2023,3,19.0,64.0,14274,4.483677
3,2023,4,17.4,171.0,22041,7.758269
4,2023,5,16.1,333.0,34403,9.679388


In [92]:
# Seleccionem les columnes que acabem de crear
df_habitatge_clean = df_habitatges_metriques[[col for col in df_habitatges_metriques.columns if col not in ["pisos_turistics", "poblacio_total"]]]
df_habitatge_clean.head()

,periode,codi_barri,preu_mitja_m2,pisos_turistics_1000_hab
0,2023,1,15.9,4.641895
1,2023,2,16.4,8.217498
2,2023,3,19.0,4.483677
3,2023,4,17.4,7.758269
4,2023,5,16.1,9.679388


In [93]:
# Separem els dos conjunts de dades
df_habitatge_23 = df_habitatge_clean[df_habitatge_clean["periode"] == 2023].drop(columns=["periode"])
df_habitatge_15 = df_habitatge_clean[df_habitatge_clean["periode"] == 2015].drop(columns=["periode"])

In [94]:
df_habitatges_pivot = df_habitatge_clean.pivot(index = "codi_barri", columns="periode")
df_habitatges_pivot.head()

preu_mitja_m2       pisos_turistics_1000_hab          
periode             2015  2023                     2015      2023
codi_barri                                                       
1                   11.1  15.9                 3.817603  4.641895
2                   11.2  16.4                11.860255  8.217498
3                   16.3  19.0                 4.588681  4.483677
4                   12.7  17.4                 7.610824  7.758269
5                   10.9  16.1                10.872322  9.679388

In [95]:
# Combinem noms de les columnes
df_habitatges_pivot.columns = [f"{col}_{year}" for col, year in df_habitatges_pivot.columns]
df_habitatges_pivot = df_habitatges_pivot.reset_index()
df_habitatges_pivot.head()

,codi_barri,preu_mitja_m2_2015,preu_mitja_m2_2023,pisos_turistics_1000_hab_2015,pisos_turistics_1000_hab_2023
0,1,11.1,15.9,3.817603,4.641895
1,2,11.2,16.4,11.860255,8.217498
2,3,16.3,19.0,4.588681,4.483677
3,4,12.7,17.4,7.610824,7.758269
4,5,10.9,16.1,10.872322,9.679388


In [96]:
deltes = ["preu_mitja_m2", "pisos_turistics_1000_hab"]

for delta in deltes:
    df_habitatges_pivot[f"delta_pct_{delta}"] = (df_habitatges_pivot[f"{delta}_2023"] - df_habitatges_pivot[f"{delta}_2015"]) / df_habitatges_pivot[f"{delta}_2015"]

df_habitatges_pivot.head()

,codi_barri,preu_mitja_m2_2015,preu_mitja_m2_2023,pisos_turistics_1000_hab_2015,pisos_turistics_1000_hab_2023,delta_pct_preu_mitja_m2,delta_pct_pisos_turistics_1000_hab
0,1,11.1,15.9,3.817603,4.641895,0.432432,0.215919
1,2,11.2,16.4,11.860255,8.217498,0.464286,-0.307140
2,3,16.3,19.0,4.588681,4.483677,0.165644,-0.022883
3,4,12.7,17.4,7.610824,7.758269,0.370079,0.019373
4,5,10.9,16.1,10.872322,9.679388,0.477064,-0.109722


In [97]:
# Ens quedem amb els deltas per al dataset
habitatge_final = df_habitatges_pivot[[col for col in df_habitatges_pivot.columns if col.startswith("delta") or col == "codi_barri"]]
habitatge_final.head()

,codi_barri,delta_pct_preu_mitja_m2,delta_pct_pisos_turistics_1000_hab
0,1,0.432432,0.215919
1,2,0.464286,-0.307140
2,3,0.165644,-0.022883
3,4,0.370079,0.019373
4,5,0.477064,-0.109722


# Combinacio
En aquest apartat combinem els diferents datasets que hem anat creant per crear: 
- Dataset de 2015
- Dataset de 2023
- Dataset amb deltes

In [98]:
# Creem base
barris = dim_barris[["codi_barri"]].copy()

In [99]:
# Conjunt de dades per 2015
df_2015 = barris.merge(poblacio_15, on="codi_barri", how = "left")\
                .merge(econs_15, on = "codi_barri", how = "left")\
                .merge(df_urbanes_15, on = "codi_barri", how = "left")\
                .merge(df_habitatge_15, on = "codi_barri")
df_2015.head()

,codi_barri,poblacio_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris,import_euros,index_gini,total_incidents_1000_hab,locals_restauracio_1000_hab,locals_sanitaris_1000_hab,locals_serveis_professionals_1000_hab,preu_mitja_m2,pisos_turistics_1000_hab
0,1,47150,0.480424,0.095949,0.519576,0.394040,0.184963,8507.809524,36.338095,14.294804,11.537646,0.148462,0.699894,11.1,3.817603
1,2,15514,0.419428,0.209037,0.580572,0.428129,0.333570,11447.444444,41.087500,40.286193,33.324739,0.257832,0.322290,11.2,11.860255
2,3,15037,0.312961,0.151160,0.687039,0.373213,0.214737,10818.818182,34.954545,40.899116,14.497573,0.532021,0.399016,16.3,4.588681
3,4,22468,0.392736,0.194632,0.607264,0.403819,0.323393,11689.000000,39.515385,26.348585,16.645896,0.356062,0.979170,12.7,7.610824
4,5,31548,0.194339,0.076455,0.805661,0.296215,0.313966,15441.500000,34.020000,17.877520,5.515405,0.507164,1.743375,10.9,10.872322


In [100]:
# Conjunt de dades per 2015
df_2023 = barris.merge(poblacio_23, on="codi_barri", how = "left")\
                .merge(econs_23, on = "codi_barri", how = "left")\
                .merge(df_urbanes_23, on = "codi_barri", how = "left")\
                .merge(df_habitatge_23, on = "codi_barri")
df_2023.head()

,codi_barri,poblacio_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris,import_euros,index_gini,total_incidents_1000_hab,locals_restauracio_1000_hab,locals_sanitaris_1000_hab,locals_serveis_professionals_1000_hab,preu_mitja_m2,pisos_turistics_1000_hab
0,1,45671,0.509623,0.106413,0.490377,0.381314,0.241860,11917.476190,33.795238,16.378008,10.947866,0.175166,0.218957,15.9,4.641895
1,2,24460,0.657195,0.146648,0.342805,0.503434,0.278250,16013.222222,39.266667,28.618152,17.988553,0.245298,0.081766,16.4,8.217498
2,3,14274,0.421045,0.192518,0.578955,0.408435,0.305240,15359.818182,32.809091,45.607398,12.400168,1.050862,0.350287,19.0,4.483677
3,4,22041,0.466267,0.212059,0.533733,0.420580,0.406288,16621.153846,36.769231,30.670115,17.558187,0.589810,0.090740,17.4,7.758269
4,5,34403,0.321164,0.104526,0.678836,0.325088,0.377961,20418.800000,31.890000,18.544894,5.813447,0.668546,0.959219,16.1,9.679388


In [101]:
# Conjunt de dades per 2015
df_deltes = barris.merge(poblacio_final, on="codi_barri", how = "left")\
                .merge(econs_final, on = "codi_barri", how = "left")\
                .merge(urbanes_final, on = "codi_barri", how = "left")\
                .merge(habitatge_final, on = "codi_barri")
df_deltes.head()

,codi_barri,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_pob_pct,delta_renda_pct,delta_gini_pct,delta_pct_total_incidents_1000_hab,delta_pct_locals_restauracio_1000_hab,delta_pct_locals_sanitaris_1000_hab,delta_pct_locals_serveis_professionals_1000_hab,delta_pct_preu_mitja_m2,delta_pct_pisos_turistics_1000_hab
0,1,0.029199,0.010464,-0.012726,0.056897,-0.031368,0.400769,-0.069978,0.145732,-0.051118,0.179867,-0.687156,0.432432,0.215919
1,2,0.237768,-0.062389,0.075305,-0.055319,0.576640,0.398847,-0.044316,-0.289629,-0.460204,-0.048610,-0.746296,0.464286,-0.307140
2,3,0.108084,0.041357,0.035222,0.090503,-0.050742,0.419732,-0.061378,0.115119,-0.144673,0.975226,-0.122122,0.165644,-0.022883
3,4,0.073531,0.017427,0.016761,0.082895,-0.019005,0.421948,-0.069496,0.164014,0.054806,0.656481,-0.907330,0.370079,0.019373
4,5,0.126825,0.028071,0.028873,0.063995,0.090497,0.322333,-0.062610,0.037330,0.054038,0.318206,-0.449792,0.477064,-0.109722


In [102]:
# Exportem datasets
df_2015.to_csv("../data/modelling/df_2015.csv")
df_2023.to_csv("../data/modelling/df_2023.csv")
df_deltes.to_csv("../data/modelling/df_deltes.csv")